In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.signal import find_peaks


# === Core Functions ===


def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

"""def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
        
    nonzero_sum = np.sum(np.where(X != 0, X, 0), axis=0)
    nonzero_count = np.count_nonzero(X, axis=0)
    
    # Avoid division by zero: set average to 0 where count is 0
    with np.errstate(divide='ignore', invalid='ignore'):
        global_spectrum = np.true_divide(nonzero_sum, nonzero_count)
        global_spectrum[nonzero_count == 0] = 0

    return mz_axis, global_spectrum"""

"""def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):
        X = X.toarray()
    global_spectrum = np.mean(X, axis=0)  # use average instead of sum
    return mz_axis, global_spectrum"""
"""
def detect_peaks(mz_axis, global_spectrum, 
                 min_prominence=0.02, top_n=None,
                 da=0.01,
                 save_path="detected_peaks.txt"):
    
    # Step 1: Peak detection (prominence-based)
    peaks, _ = find_peaks(global_spectrum, prominence=min_prominence)
    print(f"Number of detected peaks: {len(peaks)}")
    
    # Step 2: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(global_spectrum[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    # Step 3: Extract m/z and intensity
    mz_peaks = mz_axis[peaks]
    intensities = global_spectrum[peaks]

    # Step 4: Add nearby peak intensities within ±da
    adjusted_intensities = []
    for i, mz in enumerate(mz_peaks):
        # Find indices in the full mz_axis within ±da range
        mask = np.abs(mz_axis - mz) <= da
        summed_intensity = np.sum(global_spectrum[mask])
        adjusted_intensities.append(summed_intensity)

        # Find indices within ±da range
        '''mask = np.abs(mz_peaks - mz) <= da
        summed_intensity = np.sum(intensities[mask])
        adjusted_intensities.append(summed_intensity)'''
        
    adjusted_intensities = np.array(adjusted_intensities)

    # Step 5: Sort by summed intensity
    sorted_indices = np.argsort(adjusted_intensities)[::-1]
    mz_sorted = mz_peaks[sorted_indices]
    intensity_sorted = adjusted_intensities[sorted_indices]

    # Step 6: Save to text file
    np.savetxt(save_path, np.column_stack((mz_sorted, intensity_sorted)), 
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return mz_sorted, intensity_sorted
    """

def detect_peaks(mz_axis, global_spectrum, 
                 min_prominence=0.02, top_n=None,
                 da=0.01,
                 save_path="detected_peaks.txt"):
    
    # Step 1: Peak detection
    peaks, _ = find_peaks(global_spectrum, prominence=min_prominence)
    print(f"Initial detected peaks: {len(peaks)}")

    mz_peaks = mz_axis[peaks]
    intensities = global_spectrum[peaks]

    # Step 2: Sort peaks by intensity descending
    sorted_indices = np.argsort(intensities)[::-1]
    mz_peaks_sorted = mz_peaks[sorted_indices]
    intensities_sorted = intensities[sorted_indices]

    # Step 3: Filter out peaks that are too close (within ±da of a stronger one)
    final_mz = []
    final_intensity = []
    for i, mz in enumerate(mz_peaks_sorted):
        # Check if already covered by a previously accepted peak
        if not any(np.abs(mz - accepted_mz) <= da for accepted_mz in final_mz):
            final_mz.append(mz)
            final_intensity.append(intensities_sorted[i])

    final_mz = np.array(final_mz)
    final_intensity = np.array(final_intensity)

    print(f"Filtered peaks (unique by ±{da} Da): {len(final_mz)}")

    # Step 4: Optional: Top-N filtering after proximity filtering
    if top_n is not None and len(final_mz) > top_n:
        top_indices = np.argsort(final_intensity)[-top_n:][::-1]
        final_mz = final_mz[top_indices]
        final_intensity = final_intensity[top_indices]

    # Step 5: Save to file
    np.savetxt(save_path, np.column_stack((final_mz, final_intensity)),
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return final_mz, final_intensity


"""

def detect_peaks(mz_axis, global_spectrum, 
                           min_prominence=0.02, top_n=None,
                           save_path="detected_peaks.txt"):
    
    # Step 3: Peak detection (prominence-based)
    peaks, _ = find_peaks(global_spectrum, prominence=min_prominence)

    # Step 4: Optional: Top-N peaks
    if top_n is not None and len(peaks) > top_n:
        top_indices = np.argsort(global_spectrum[peaks])[-top_n:]
        peaks = peaks[np.sort(top_indices)]

    # Extract m/z and intensity
    mz_peaks = mz_axis[peaks]
    intensities = global_spectrum[peaks]

    # Sort by intensity (descending)
    sorted_indices = np.argsort(intensities)[::-1]
    mz_sorted = mz_peaks[sorted_indices]
    intensity_sorted = intensities[sorted_indices]

    # Save to text file
    np.savetxt(save_path, np.column_stack((mz_sorted, intensity_sorted)), 
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return mz_sorted, intensity_sorted



"""


'\n\ndef detect_peaks(mz_axis, global_spectrum, \n                           min_prominence=0.02, top_n=None,\n                           save_path="detected_peaks.txt"):\n    \n    # Step 3: Peak detection (prominence-based)\n    peaks, _ = find_peaks(global_spectrum, prominence=min_prominence)\n\n    # Step 4: Optional: Top-N peaks\n    if top_n is not None and len(peaks) > top_n:\n        top_indices = np.argsort(global_spectrum[peaks])[-top_n:]\n        peaks = peaks[np.sort(top_indices)]\n\n    # Extract m/z and intensity\n    mz_peaks = mz_axis[peaks]\n    intensities = global_spectrum[peaks]\n\n    # Sort by intensity (descending)\n    sorted_indices = np.argsort(intensities)[::-1]\n    mz_sorted = mz_peaks[sorted_indices]\n    intensity_sorted = intensities[sorted_indices]\n\n    # Save to text file\n    np.savetxt(save_path, np.column_stack((mz_sorted, intensity_sorted)), \n               fmt="%.6f", header="m/z\tintensity", delimiter="\t")\n\n    return mz_sorted, intensity_s

In [3]:
# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
#output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style_1000_0.04_100.h5ad"

# === Load and Process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

X = adata.X
X = X.toarray() if hasattr(X, "toarray") else X
row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
X_norm = X / row_sums

mz_axis, global_spec = create_global_spectrum(X, adata.var_names.astype(float).values)
#mz_axis, global_spec = create_global_spectrum_max(adata.X, adata.var_names.astype(float).values)
peak_mz, peak_intensities = detect_peaks(mz_axis, global_spec, 
                                        min_prominence=0.00001, top_n=1000,
                                        da=0.02,
                                        save_path="first_top_peaks_00001_002da_notadd_glob_1000top_avg.txt")

print(f'Number of peaks: {len(peak_mz)}')


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad
Initial detected peaks: 25194
Filtered peaks (unique by ±0.02 Da): 19747
Number of peaks: 1000
